# RAG — Retrieval Augmented Generation

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*¿Cómo le doy conocimiento propio a un LLM sin reentrenarlo?*

## Recap Clases 1-2 → conexión con hoy

```
  Clase 1                            Clase 2
  ───────                            ───────
  Texto → Tokens → Embeddings        Embeddings → Transformer → Texto generado
       BoW  TF-IDF  Vectores 384D        Attention   Prompting   llamar_llm()
                │                              │
                └──────── HOY ─────────────────┘
                     RAG une retrieval con generación
```

| Lo que ya sabemos | Lo que resolvemos hoy |
|---|---|
| Embeddings capturan significado semántico | Los usamos para **recuperar** documentos relevantes |
| TF-IDF / BM25 buscan por coincidencia de palabras | Los combinamos con búsqueda semántica |
| `llamar_llm()` genera texto con un prompt | Le **inyectamos contexto** antes de generar |
| Context window tiene límites | RAG selecciona **solo lo relevante** |

> En **Clase 3b** (siguiente sesión) vemos **cómo medir y operar** lo que construyamos hoy — eval offline, tracing en producción, drift detection, práctica con Arize AX.


## Recorrido de la clase

| Bloque | Tema | Tiempo aprox. |
|---|---|---|
| **B1** | ¿Por qué RAG? + aplicaciones típicas + LLM vs RAG | ~15 min |
| **B2** | Pipeline paso a paso + chunking + ejemplo Santa Fe | ~30 min |
| ☕ | _break_ | 10 min |
| **B3** | Búsqueda híbrida (BM25 + semántica) | ~15 min |
| **B4** | RAG avanzado: reranking, HyDE, parent-child, Graph + Multi-Hop + troubleshooting | ~30 min |

Vamos a **construir el RAG paso a paso** con demos ejecutables y ejercicios. En **Clase 3b** profundizamos en cómo **medir y operar** sistemas LLM en producción, con práctica end-to-end con Arize AX.


## ⚙️ Setup

**Las prácticas son notebooks separados** que corren en Google Colab. Cada uno
se autoinstala las dependencias y configura las credenciales al inicio.

**Dependencias usadas a lo largo de la clase** (cada notebook hace su propio
`pip install`):

```bash
pip install sentence-transformers chromadb rank_bm25 groq numpy pandas matplotlib
```

**Pre-requisitos:**
- Cuenta gratuita en Groq → https://console.groq.com/keys
- Google account para Colab (gratis)

> El código que durante la clase muestre en pantalla va a estar en los notebooks
> linkeados al final de cada bloque. Las slides son sólo conceptuales.


# Bloque 1
## ¿Por qué RAG?

---

*El LLM sabe mucho, pero no sabe lo tuyo.*

## El problema del LLM puro

### Tres limitaciones que no se resuelven con mejor prompting

| Limitación | Ejemplo | Consecuencia |
|---|---|---|
| **Knowledge cutoff** | "¿Qué pasó en las elecciones de ayer?" | El modelo no tiene datos recientes |
| **Alucinaciones** | "Citame el artículo 47 del reglamento de la UTN" | Inventa contenido con total confianza |
| **Context window** | 50.000 documentos internos de una empresa | No caben ni en 1M de tokens |

```
  Opción 1: Re-entrenar el modelo     → costoso, lento, no práctico
  Opción 2: Meter todo en el prompt   → no escala, "Lost in the Middle"
  Opción 3: RAG                       → recuperar solo lo relevante ✓
```

> **RAG = no cambiás el modelo, cambiás lo que le das para leer.**

> 📖 *Lewis, P., et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. NeurIPS 2020. https://arxiv.org/abs/2005.11401*

## RAG — Retrieval Augmented Generation

**Idea central:** antes de que el LLM responda, buscar información relevante y ponerla en el prompt.

```
  ┌────────────────┐         ┌────────────────┐         ┌─────────────────────┐
  │  Base de       │ indexar │  Vector        │  top-k  │  Prompt =           │
  │  conocimiento  │ ──────▶ │  Store         │ ──────▶ │  chunks + query     │
  └────────────────┘         └───────┬────────┘         └──────────┬──────────┘
                                     │ buscar                      │
                             ┌───────┴────────┐                    ▼
                             │     Query      │              ┌──────────┐
                             └────────────────┘              │   LLM    │
                                                             └────┬─────┘
                                                                  ▼
                                                             Respuesta
```

**Los 4 pasos:**
1. **INDEXAR:** documentos → chunks → embeddings → vector store
2. **CONSULTAR:** query → embedding → buscar top-k chunks similares
3. **AUGMENTAR:** armar prompt = system + chunks recuperados + pregunta
4. **GENERAR:** el LLM responde usando los chunks como contexto

## LLM clásico vs LLM con RAG

| Capacidad | LLM clásico | LLM + RAG |
|---|---|---|
| **Conocimiento posterior al training** | ❌ Imposible | ✅ Vía retrieval sobre fuentes recientes |
| **Conocimiento privado / propietario** | ❌ Requiere fine-tuning | ✅ Solo indexar — el modelo queda intacto |
| **Citas y trazabilidad** | ❌ Inventa fuentes o no las da | ✅ Cada respuesta se rastrea a chunks específicos |
| **Reducción de alucinaciones** | ❌ Confianza alta en respuestas inventadas | ✅ Forzado a basarse en contexto (faithfulness) |
| **Razonamiento sobre documentos largos** | ⚠️ Limitado al context window | ✅ Solo trae lo relevante (evita lost-in-the-middle) |
| **Actualización del conocimiento** | ⚠️ Re-entrenar (caro y lento) | ✅ Re-indexar (rápido, barato, incremental) |
| **Costo por consulta** | ✅ Más barato | ⚠️ Mayor (retrieval + generation) |
| **Latencia** | ✅ Menor | ⚠️ Mayor (extra retrieval step) |

> Las dos últimas filas son los **tradeoffs reales** que RAG impone. Por todo lo demás, RAG es la opción default cuando hay corpus propio.


## Aplicaciones típicas de RAG (2025-2026)

| Dominio | Caso de uso | Por qué encaja RAG | Productos reales |
|---|---|---|---|
| **Soporte / Customer success** | Q&A sobre docs de producto, helpdesks internos, triage de tickets | Base de conocimiento que cambia frecuentemente; respuesta con citas | Intercom Fin, Glean, Notion AI |
| **Legal** | Búsqueda de jurisprudencia, revisión de contratos, compliance | Citas obligatorias; vocabulario muy técnico; corpus enorme | Harvey, Casetext CoCounsel, Eve.legal |
| **Salud / Medicina** | Decisión clínica sobre papers y guías; resumen de historia clínica | Knowledge cutoff es crítico (papers nuevos); auditabilidad regulatoria | Glass, OpenEvidence, Hippocratic AI |
| **Finanzas / Regulatorio** | Q&A sobre filings (10-K/10-Q), due diligence, riesgo regulatorio | Documentos largos y técnicos; toda respuesta debe rastrearse | Hebbia, AlphaSense, BloombergGPT |
| **Ingeniería / Código** | Q&A sobre codebase, runbooks, postmortems, generación de tests | Repos grandes, contexto distribuido en muchos archivos | Sourcegraph Cody, Cursor, Sentry Autofix |
| **Educación** | Tutor sobre apuntes/libros, asistente de cursada, prep de examen | Cada estudiante consulta el mismo material desde ángulos distintos | Khanmigo, CourseHero, Speak |
| **Investigación científica** | Literature review, búsqueda semántica, generación de hipótesis | Volumen masivo (millones de papers); terminología específica | Elicit, Consensus, Scite |
| **Enterprise / Knowledge ops** | "Ask the company": wikis, políticas, HR, runbooks internos | Datos privados que no entran en el training de ningún LLM público | Glean, Notion AI, Slack AI |

### Patrón común a todos
**(1) corpus propio o nicho** + **(2) requerimiento de citas/groundedness** + **(3) actualización frecuente**.

### Cuándo RAG *no* es la respuesta
- Tareas **sin necesidad de conocimiento externo** (traducción, redacción creativa, conversación general).
- Datos que **caben en el context window** (< 200K tokens) → mandalos directo, sin retrieval.
- Tareas donde lo que falta es **razonamiento estructurado**, no información (cálculo numérico exacto, código complejo).


# Bloque 2
## Pipeline RAG naive

---

*De documentos sueltos a respuestas fundamentadas, paso a paso.*

## El pipeline RAG paso a paso

```
  FASE 1: INDEXACIÓN (se hace una vez)
  ─────────────────────────────────────────────────────────────────
  Documentos  ──▶  Chunking  ──▶  Embeddings  ──▶  Vector Store
                                                        │
                                                        │ (persiste)
  FASE 2: CONSULTA (cada pregunta)                      │
  ─────────────────────────────────────────────────────────────────
  Query  ──▶  Embedding  ──▶  Retrieval  ◀─────────────┘
                                  │
                                  ▼
                           Top-k chunks
                                  │
                                  ▼
                    Augmentation (prompt = chunks + query)
                                  │
                                  ▼
                                LLM  ──▶  Respuesta
```

Hoy implementamos cada paso de este diagrama.

## Chunking — partir documentos en fragmentos

### ¿Por qué? Un documento de 10 páginas no cabe como un solo embedding.

| Estrategia | Cómo funciona | Cuándo usarla |
|---|---|---|
| **Tamaño fijo** | Cada N caracteres, con overlap | Default simple, documentos homogéneos |
| **Por oración** | Partir en `. ` `? ` `! ` | Cuando cada oración es autocontenida |
| **Por párrafo** | Partir en `\n\n` | Documentos bien estructurados |
| **Recursivo** | Probar `\n\n` → `\n` → `. ` → espacio | LangChain default, el más robusto |

```
  Chunking fijo (200 chars, overlap 50):
  ┌──────────────┐
  │  Chunk 1     │
  │  chars 0-199 │
  └──────┬───────┘
         │overlap│
         ┌───────┴──────┐
         │  Chunk 2     │
         │  chars 150-349│
         └──────────────┘
```

**Overlap:** repetir N caracteres entre chunks para no cortar ideas a la mitad.  
Regla práctica: overlap = 10-20% del tamaño del chunk.

## Tamaño de chunk — el knob más importante

| Tamaño | Precisión | Contexto | Costo / Latencia | Cuándo usarlo |
|---|---|---|---|---|
| **64-128 tokens** | 🟢 Alta | 🔴 Bajo | 🟢 Bajo | Q&A factual corta, lookups exactos |
| **256-512 tokens** | 🟡 Media | 🟡 Medio | 🟡 Medio | **Default — el sweet spot** para la mayoría |
| **1024+ tokens** | 🔴 Baja | 🟢 Alto | 🔴 Alto | Documentos legales/técnicos con razonamiento largo |

### Reglas de bolsillo

```
  ┌────────────────────────────────────────────────────────────────────┐
  │ overlap     = 10-20% del tamaño del chunk                          │
  │ k           = 5-10 (cuántos chunks recuperás)                      │
  │ contexto    = k × chunk_size + prompt + query ≤ 50% del window     │
  └────────────────────────────────────────────────────────────────────┘
```

### Cómo ajustarlo en la práctica
1. Empezá con **chunk=512, overlap=50, k=5** y un eval offline mínimo.
2. Si faltan detalles en la respuesta → **bajá chunk, subí k**.
3. Si la respuesta pierde coherencia entre chunks → **subí chunk, mantené k**.
4. Si latencia o costo duelen → **bajá k antes de bajar chunk**.

> No hay una respuesta universal. **Es un parámetro que se afina con eval offline** (lo vemos en B5).


## ChromaDB — vector store en memoria

### ¿Qué es un vector store?

Una base de datos especializada en guardar vectores y buscar los más similares.

```
  Base relacional (SQL):              Vector store:
  ─────────────────────               ─────────────
  SELECT * FROM docs                  "buscar los 3 chunks
  WHERE categoria = 'IA'              más parecidos a esta query"
  → coincidencia EXACTA               → similitud APROXIMADA

  Busca por: igualdad, rango          Busca por: distancia en espacio N-dim
  Índice: B-tree                      Índice: HNSW
```

**ChromaDB:**
- Open source, Python nativo
- Funciona **in-memory** — sin instalar nada
- Perfecto para prototipos y clases
- En producción: Pinecone, Weaviate, Qdrant, pgvector

| Config | Valor que usamos | Por qué |
|---|---|---|
| Distancia | `cosine` | Misma métrica que Clase 1 |
| Embedding function | `sentence-transformers` manual | Control total, ya lo conocemos |
| Metadata | `titulo`, `doc_id`, `chunk_index` | Para saber de dónde viene cada chunk |

## Augmentation — el prompt que une todo

### El paso clave: construir un prompt que combine chunks recuperados con la pregunta.

```
  ┌─────────────────────────────────────────────────┐
  │  SYSTEM                                         │
  │  "Sos un asistente que responde basándose       │
  │   SOLO en el contexto proporcionado.            │
  │   Si no hay info suficiente, decilo."           │
  ├─────────────────────────────────────────────────┤
  │  USER                                           │
  │                                                 │
  │  Contexto:                                      │
  │  [1] (Arquitectura) "Los modelos de ML se..."   │
  │  [2] (MLOps) "MLOps aplica prácticas de..."     │
  │  [3] (MLOps) "Los contenedores Docker con..."   │
  │                                                 │
  │  Pregunta: ¿Cómo se despliegan modelos en prod? │
  ├─────────────────────────────────────────────────┤
  │  ASSISTANT                                      │
  │  "Según los documentos, los modelos se..."      │
  └─────────────────────────────────────────────────┘
```

**Reglas del prompt RAG:**
1. El system prompt **restringe** al LLM a usar solo el contexto dado
2. Los chunks se numeran para que el LLM pueda citar fuentes
3. La pregunta va **al final** (recency bias: el modelo presta más atención al final)

## Ejemplo end-to-end — multas de tránsito en Santa Fe

**Usuario:** *"¿Cuál es el plazo de prescripción para multas de tránsito en la Ciudad de Santa Fe?"*

```
  ┌─────────────────────────────────────────────────────────────────────┐
  │ 1. INDEXACIÓN (offline, se hace una vez)                             │
  │    • Código de Faltas + Resoluciones + Jurisprudencia local          │
  │    • Chunking semántico (~400 tokens, overlap 50)                    │
  │    • Embeddings multilingüe → ChromaDB                               │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 2. RETRIEVAL (online, por query)                                     │
  │    • Embed de la pregunta del usuario                                │
  │    • Top-5 chunks: Art. 47 (prescripción), Art. 52 (cómputo de plazo),│
  │      resumen de jurisprudencia 2023 sobre suspensión del plazo       │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 3. AUGMENTATION                                                      │
  │    • Prompt = system("citá fuentes") + chunks + query                │
  ├─────────────────────────────────────────────────────────────────────┤
  │ 4. GENERACIÓN                                                        │
  │    • "Según el Art. 47 del Código de Faltas de Santa Fe,             │
  │       la prescripción es de 2 años desde la notificación.            │
  │       El plazo se suspende durante el proceso administrativo         │
  │       (Art. 52). Fuente: [Art. 47, Código de Faltas]"                │
  └─────────────────────────────────────────────────────────────────────┘
```

### Por qué este caso es **paradigmático** para RAG
- Datos **públicos pero dispersos** (no entran en el training de ningún LLM global).
- **Auditabilidad obligatoria** — la respuesta debe rastrearse al artículo (no es opcional en lo legal).
- **Actualización constante** — los reglamentos cambian; re-entrenar el LLM no es viable.
- Dominio **localizado** que ningún modelo genérico cubre con la precisión necesaria.


## 🛠 Práctica del Bloque 2 — Pipeline RAG naive

📓 [`clase03/notebooks/rag/01_naive.ipynb`](notebooks/rag/01_naive.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/01_naive.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lo que vas a hacer (~30 min):**
- Demo "LLM sin contexto vs con contexto" (motivación).
- Levantar el corpus de 5 docs sobre IA para ingeniería de sistemas.
- Chunkear, embeddings, ChromaDB.
- Construir `rag_query()` end-to-end.
- **Correr el benchmark de 7 queries con RAG naive y completar la tabla de resultados.** Esta es la **baseline** que las notebooks 2 y 3 van a tratar de mejorar.

> El benchmark cubre: fáciles · ambiguas (sigla vs concepto) · multi-hop · edge case.


# Bloque 3
## Hybrid Search

---

*A veces necesitás palabras exactas. A veces necesitás significado. Lo ideal: ambos.*

## BM25 — recap rápido

En Clase 1 vimos BoW y TF-IDF. **BM25** es la evolución de TF-IDF para búsqueda:

```
  BM25(query, doc) = Σ  IDF(término) × TF_saturada(término, doc) × norm_largo
                    término∈query
```

Mejoras sobre TF-IDF:
- **Saturación de TF:** la 10ma aparición de una palabra no suma tanto como la 1ra
- **Normalización por largo:** docs cortos no se penalizan injustamente

### ¿Cuándo falla la búsqueda semántica?

| Query | Búsqueda semántica | BM25 |
|---|---|---|
| "ChromaDB" | Busca conceptos similares a DBs vectoriales ✓ | Busca la palabra exacta "ChromaDB" ✓✓ |
| "HNSW" | No entiende la sigla, devuelve ruido ✗ | Encuentra el chunk que dice "HNSW" ✓ |
| "seguridad en IA" | Entiende el concepto amplio ✓✓ | Solo matchea si dice "seguridad" literal ✓ |
| "cómo protegerse de ataques" | Entiende la intención ✓✓ | No matchea "prompt injection" ✗ |

**Conclusión:** BM25 gana para términos técnicos y siglas. Semántica gana para conceptos. Lo ideal: combinarlos.

## 🛠 Práctica del Bloque 3 — Hybrid Search

📓 [`clase03/notebooks/rag/02_hybrid.ipynb`](notebooks/rag/02_hybrid.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/02_hybrid.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lo que vas a hacer (~25 min):**
- Implementar BM25 standalone sobre los mismos chunks.
- Implementar Hybrid Search con weighted sum (α).
- **Correr el mismo benchmark de la notebook 1 con dense / BM25 / hybrid** y comparar fila por fila.
- Variar α (0.0, 0.3, 0.5, 0.7, 1.0) para ver qué valor funciona mejor por tipo de query.


# Bloque 4
## RAG avanzado

---

*El pipeline naive funciona. Estas técnicas lo hacen funcionar mejor.*

## Reranking — segunda pasada de precisión

### Problema: el retrieval trae top-k, pero el orden no siempre es óptimo.

```
  Pipeline naive:                    Con reranking:
  ───────────────                    ──────────────
  Query → Vector Store → Top 10     Query → Vector Store → Top 10
                          ↓                                  ↓
                        LLM ✗                        Cross-encoder
                                                     reordena → Top 3
                                                          ↓
                                                        LLM ✓
```

**Cross-encoder vs Bi-encoder:**

| | Bi-encoder (lo que usamos) | Cross-encoder (reranker) |
|---|---|---|
| Input | Query y doc por separado | Query + doc juntos |
| Velocidad | Rápido (embeddings pre-calculados) | Lento (procesa cada par) |
| Precisión | Buena | Mejor |
| Uso | Retrieval inicial (miles de docs) | Reranking (top 10-20 candidatos) |

```
  Bi-encoder:    embed(query)  →  comparar con  ←  embed(doc)   [rápido, menos preciso]
  Cross-encoder: model(query + doc)  →  score de relevancia     [lento, más preciso]
```

Modelo popular: `cross-encoder/ms-marco-MiniLM-L-6-v2` (~80MB)

## HyDE — Hypothetical Document Embeddings

### Problema: la query del usuario es corta; los documentos son largos.

El embedding de "cómo monitorear un modelo" no se parece al embedding de un párrafo técnico sobre MLOps.

**Idea:** pedirle al LLM que **invente** un documento hipotético que responda la pregunta, y buscar con ESE embedding.

```
  Pipeline normal:     Query → embed(query) → buscar → chunks

  Pipeline HyDE:       Query → LLM genera doc hipotético
                                    ↓
                              embed(doc_hipotético) → buscar → chunks
```

**¿Por qué funciona?** El embedding del documento hipotético está más cerca del "espacio" de los documentos reales que el embedding de la query corta.

**Costo:** una llamada extra al LLM por cada query (latencia + tokens).

> 📖 *Gao, L., et al. (2023). Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE). ACL 2023. https://arxiv.org/abs/2212.10496*

## Parent-child chunks — contexto completo

### Problema: chunks chicos = retrieval preciso pero sin contexto. Chunks grandes = retrieval impreciso.

**Solución:** indexar chunks chicos (hijos) pero recuperar el chunk grande (padre).

```
  Documento original:
  ┌────────────────────────────────────────────────────┐
  │  Párrafo completo sobre seguridad en IA (padre)    │
  │  ┌──────────┐ ┌──────────┐ ┌──────────┐           │
  │  │ Oración 1│ │ Oración 2│ │ Oración 3│  (hijos)  │
  │  │ (indexar) │ │ (indexar) │ │ (indexar) │           │
  │  └──────────┘ └──────────┘ └──────────┘           │
  └────────────────────────────────────────────────────┘

  Query → match con Oración 2 (hijo)
       → devolver Párrafo completo (padre)
       → el LLM tiene más contexto para responder
```

**Implementación:** cada chunk hijo guarda un `parent_id` en metadata.  
Al recuperar, se busca por hijo pero se devuelve el padre.

**Trade-off:**
- Más contexto para el LLM → mejor respuesta
- Más tokens por chunk → más costo, posible ruido

> Esta técnica se implementa con LangChain `ParentDocumentRetriever`. La vemos en detalle en Clase 4 si hace falta para el TP Integrador.

## Graph RAG y Multi-Hop RAG — más allá del top-k

### Multi-Hop RAG — razonamiento sobre varios documentos

```
  Query: "Compará cómo evolucionó la ley X en Argentina vs Brasil entre 2020-2025"

  Naive RAG:                       Multi-Hop:
  ──────────                       ─────────
  1 embed(query)                   Paso 1: retrieval sobre "ley X Argentina"
  2 top-5 chunks                   Paso 2: retrieval sobre "ley X Brasil"
  3 (chunks dispersos              Paso 3: retrieval sobre "evolución 2020-2025"
     no alcanzan)                  Paso 4: síntesis con todos los chunks
```

### Graph RAG — knowledge graph como índice (Microsoft, 2024)

```
  Documentos                Graph                    Query
  ─────────                 ─────                    ─────
  ┌──────────┐  extract    [Persona A] ──trabaja──▶ [Empresa X]
  │ Doc 1    │ ──con LLM──▶                              │
  ├──────────┤              [Persona A] ──colaboró──▶ [Persona B]
  │ Doc 2    │                                              │
  ├──────────┤              [Empresa X] ──proyecto──▶ [Tema IA]
  │ Doc 3    │
  └──────────┘   "¿Quién trabajó con A en X en proyectos de IA?"
                  → traverse: A → X → proyectos → IA → B
```

### Cuándo usarlas

| Técnica | Caso ideal | Cuándo NO |
|---|---|---|
| **Multi-Hop** | Preguntas que combinan info de ≥ 2 fuentes distintas | Q&A de un solo concepto, lookups simples |
| **Graph RAG** | Dominios **entidad-céntricos** (legal, biomedicina, investigación) donde las **relaciones** importan tanto como los textos | Corpus principalmente narrativo o documental |

> 📖 *Edge, D., et al. (2024). From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* https://arxiv.org/abs/2404.16130


## Naive vs Hybrid vs Advanced RAG — recap

| Componente | **Naive** | **Hybrid** | **Advanced** |
|---|---|---|---|
| **Chunking** | Tamaño fijo, sin overlap | Semántico + overlap | Jerárquico (parent-child) |
| **Retrieval** | Solo dense (embeddings) | Dense + sparse (BM25) | Multi-hop, graph traversal |
| **Reranking** | Ninguno | Cross-encoder | Cross-encoder + scoring global |
| **Fusión de fuentes** | No | Sí (RRF, weighted) | Múltiples retrievers + voto |
| **Query enrichment** | No | Query expansion básica | HyDE, query rewriting con LLM |
| **Memoria conversacional** | No | Parcial | Completa (multi-turn coherente) |
| **Costo / Latencia** | 🟢 Bajo | 🟡 Medio | 🔴 Alto |
| **Cuándo elegirlo** | Prototipo, corpus chico | Producto en serio | Dominios complejos (legal, médico) |

> **Regla de oro:** empezá *siempre* por Naive. Subí complejidad sólo cuando una métrica concreta (faithfulness, recall@k, latencia) lo justifique. El bloque 5 te enseña cómo medir.


## Problemas comunes en RAG — guía de troubleshooting

| Síntoma observable | Causa probable | Mitigación |
|---|---|---|
| Top-k trae chunks **irrelevantes** | Embedding model mal elegido; distribución de queries ≠ corpus | Hybrid search (BM25 + dense), reranking con cross-encoder |
| Respuestas que **ignoran** los chunks | Prompt débil, alucinación del LLM | Faithfulness eval, prompt con "basate SOLO en…", structured output |
| **Recall bajo** — falta info que sí existe | Chunks muy chicos; embedding sub-óptimo; falta query rewriting | Parent-child chunks, HyDE, query expansion |
| **Latencia inaceptable** | Reranking sobre demasiados candidatos; embeddings no cacheados | Reranking sólo top-20; cache de queries frecuentes; embedding quantization |
| **Costo alto** | Sin caching; embedding model caro; sobre-recuperación | Semantic cache; modelos OSS para embeddings; ajustar `k` |
| **Datos desactualizados** | Re-indexación manual; sin pipeline de ingesta | Pipeline incremental con CDC; versionar chunks; alerta de staleness |
| **Multi-documento falla** | Naive retrieval no enlaza chunks disjuntos | Multi-hop retrieval, graph RAG, agentic retrieval |
| **Vocabulario de dominio** mal entendido | Embedding genérico no captura jerga técnica | Fine-tuned embeddings (BGE, GTE); glosario de términos; query rewriting |

> Estos son **diagnósticos**, no recetas. Cada caso requiere medir si la mitigación realmente mueve la métrica (B5).


## 🛠 Prácticas del Bloque 4 — RAG Avanzado + Capstone

### Notebook 3: técnicas avanzadas

📓 [`clase03/notebooks/rag/03_advanced.ipynb`](notebooks/rag/03_advanced.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/03_advanced.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementar reranking (cross-encoder), HyDE y parent-child chunks. Correr el **mismo benchmark** con cada técnica + una combinada (~30 min).

### Notebook 4: capstone — la evolución de una query

📓 [`clase03/notebooks/rag/04_capstone.ipynb`](notebooks/rag/04_capstone.ipynb)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase03/notebooks/rag/04_capstone.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Una sola query multi-hop difícil corre por **6 stages** (LLM puro → naive → hybrid → reranking → HyDE → parent-child). Tabla evolutiva + gráfico de scores. Cierre conceptual de la práctica de RAG (~20 min).


## Lo que vimos hoy

| Bloque | Concepto clave | Conexión con el curso |
|---|---|---|
| B1 — ¿Por qué RAG? | Knowledge cutoff, alucinaciones, context limits; aplicaciones reales (legal/salud/código/etc.) | Resuelve lo que Clase 2 dejó pendiente |
| B2 — Pipeline naive | Chunking → embeddings → ChromaDB → retrieval → augmentation; ejemplo Santa Fe | Base del TP Integrador |
| B3 — Hybrid search | BM25 (léxico) + semántico (vectores) combinados | Extiende BoW/TF-IDF de Clase 1 |
| B4 — RAG avanzado | Reranking, HyDE, parent-child, Graph/Multi-Hop, troubleshooting | Técnicas para mejorar calidad |

**Stack que usamos hoy:**
```
sentence-transformers · chromadb · rank_bm25 · groq · cross-encoder
```

**Lo que sigue siendo válido para Clase 3b y 4:**
- `llamar_llm()` + `rag_query()` + `hybrid_search()` — los usamos en la práctica de B5 (Arize) y en el TP integrador de Clase 4.


---

## 📚 Bibliografía — RAG

### Libros de referencia
- Jurafsky, D. & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). Cap. 14 — Question Answering. https://web.stanford.edu/~jurafsky/slp3/

### Papers fundamentales
- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS. https://arxiv.org/abs/2005.11401
- Robertson, S. & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.*
- Gao, L. et al. (2023). *HyDE — Precise Zero-Shot Dense Retrieval without Relevance Labels.* ACL. https://arxiv.org/abs/2212.10496
- Liu, N. F. et al. (2023). *Lost in the Middle — How Language Models Use Long Contexts.* TACL. https://arxiv.org/abs/2307.03172
- Edge, D. et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* https://arxiv.org/abs/2404.16130

### Documentación / herramientas
- **ChromaDB** — https://docs.trychroma.com/
- **Sentence-Transformers** — https://www.sbert.net/
- **rank_bm25** — https://github.com/dorianbrown/rank_bm25
- **Sentence-Transformers cross-encoders** — https://www.sbert.net/examples/applications/cross-encoder/README.html

> La bibliografía completa de **eval, monitoring y benchmarks** (RAGAS, MT-Bench, SelfCheckGPT, PAIR, HELM, Arize, OpenTelemetry, etc.) vive en **Clase 3b**.


## Clase 3b — ¿qué viene?

Hoy construimos el RAG. En la **próxima sesión (Clase 3b)** vemos cómo **medir y operar** sistemas LLM en producción — es la otra mitad del problema, y se aplica tanto a RAG como a cualquier sistema con LLMs.

---

- **Fundamentos** — por qué eval de LLMs es distinto, reference-based vs free, detección de alucinaciones, tracing.
- **Pre-deploy (eval offline)** — golden datasets, LLM-as-judge, RAGAS deep dive, safety + red-teaming, armar tu propio benchmark.
- **Deploy** — A/B, shadow, canary, prompt versioning, model registry, auto-rollback.
- **Producción** — logging, cost/latency, drift detection, feedback loops.
- **Práctica end-to-end con Arize AX** — un chatbot Q&A instrumentado de punta a punta sobre los temas de las clases 1-3.

> Después de Clase 3b: **Clase 4 = Agentes**.
